# Monte-Carlo Tree Search

## Obiettivo

La **Monte-Carlo Tree Search**, o MCTS, costruisce progressivamente un albero di ricerca.
Ogni iterazione contiene quattro fasi:

1. selezione;
2. espansione;
3. simulazione;
4. backpropagation.

L'esempio usa lo stesso gioco numerico: partire da `0` e raggiungere esattamente `10`
aggiungendo `1`, `2` oppure `3`.

In [1]:
import math
import random

random.seed(42)

TARGET = 10
ACTIONS = [1, 2, 3]

In [2]:
def transition(state, action):
    """Restituisce nuovo stato, ricompensa e indicatore terminale."""
    next_state = state + action

    if next_state == TARGET:
        return next_state, 1.0, True

    if next_state > TARGET:
        return next_state, -1.0, True

    return next_state, 0.0, False

In [3]:
class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

        self.children = []
        self.visits = 0
        self.total_reward = 0.0

        # All'inizio, tutte le azioni sono ancora da provare.
        self.untried_actions = ACTIONS.copy()

    @property
    def mean_reward(self):
        if self.visits == 0:
            return 0.0
        return self.total_reward / self.visits

    def is_terminal(self):
        return self.state >= TARGET

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, exploration_weight=1.4):
        """Seleziona il figlio con formula UCB1."""
        def ucb_score(child):
            exploitation = child.mean_reward
            exploration = exploration_weight * math.sqrt(
                math.log(self.visits) / child.visits
            )
            return exploitation + exploration

        return max(self.children, key=ucb_score)

In [4]:
def select(node):
    """Scende nell'albero finché trova un nodo espandibile o terminale."""
    while not node.is_terminal():
        if not node.is_fully_expanded():
            return node
        node = node.best_child()
    return node

In [5]:
def expand(node):
    """Aggiunge un nuovo figlio usando un'azione non ancora provata."""
    action = node.untried_actions.pop()
    next_state, _, _ = transition(node.state, action)

    child = Node(
        state=next_state,
        parent=node,
        action=action,
    )
    node.children.append(child)
    return child

In [6]:
def simulate(state):
    """Esegue un rollout casuale dal nodo selezionato."""
    current_state = state

    while True:
        if current_state == TARGET:
            return 1.0
        if current_state > TARGET:
            return -1.0

        action = random.choice(ACTIONS)
        current_state, reward, done = transition(
            current_state,
            action,
        )

        if done:
            return reward

In [7]:
def backpropagate(node, reward):
    """Propaga il risultato della simulazione fino alla radice."""
    while node is not None:
        node.visits += 1
        node.total_reward += reward
        node = node.parent

In [8]:
def mcts_search(initial_state, iterations=1000):
    root = Node(initial_state)

    for _ in range(iterations):
        # 1. Selezione
        node = select(root)

        # 2. Espansione
        if not node.is_terminal():
            node = expand(node)

        # 3. Simulazione
        reward = simulate(node.state)

        # 4. Backpropagation
        backpropagate(node, reward)

    # Per la decisione finale si sceglie il figlio più visitato.
    best = max(root.children, key=lambda child: child.visits)
    return best.action, root

## Esecuzione della ricerca

In [9]:
best_action, root = mcts_search(initial_state=0, iterations=2000)

print("Azione scelta:", best_action)
print("\nStatistiche dei figli della radice:")

for child in sorted(root.children, key=lambda node: node.action):
    print(
        f"Azione={child.action}, "
        f"visite={child.visits}, "
        f"ricompensa media={child.mean_reward:.3f}"
    )

Azione scelta: 2

Statistiche dei figli della radice:
Azione=1, visite=17, ricompensa media=-0.176
Azione=2, visite=1964, ricompensa media=0.960
Azione=3, visite=19, ricompensa media=-0.158


## Decisioni successive

In [10]:
state = 0
trajectory = [state]

while state < TARGET:
    action, root = mcts_search(
        initial_state=state,
        iterations=1000,
    )

    state, reward, done = transition(state, action)
    trajectory.append(state)

    print(
        f"Azione={action}, nuovo stato={state}, ricompensa={reward}"
    )

    if done:
        break

print("\nTraiettoria:", trajectory)

Azione=2, nuovo stato=2, ricompensa=0.0
Azione=3, nuovo stato=5, ricompensa=0.0
Azione=3, nuovo stato=8, ricompensa=0.0
Azione=2, nuovo stato=10, ricompensa=1.0

Traiettoria: [0, 2, 5, 8, 10]


## Differenza rispetto alla ricerca Monte-Carlo semplice

La ricerca Monte-Carlo semplice valuta direttamente le azioni iniziali tramite rollout.
MCTS, invece, concentra progressivamente le simulazioni sui rami più promettenti e
conserva statistiche per tutti i nodi visitati.